# Taller 2. Reportes corporativos colombianos
## Del informe de fin de ejercicio (PDF) a la nota de valoración asistida por IA

Este cuaderno es la **versión para emisores colombianos** del flujo de la sesión 8
(*Del reporte corporativo a la valoración de empresas*) y sirve de plantilla para el **Taller 2
(evaluable)**. La lógica es la misma: segmentar, medir tono y cambio, preguntar con RAG, extraer y
verificar, y construir un DCF por escenarios. Lo que cambia es **todo lo que dependía de la SEC y del
inglés**:

| En la sesión 8 (EE. UU.) | En este cuaderno (Colombia) |
|---|---|
| 10-K en HTML descargado de EDGAR | **PDF** del informe periódico de fin de ejercicio o del informe de gestión |
| Secciones fijas: *Item 1A* e *Item 7* | No hay *Items*: tú indicas las **páginas** de *riesgos* y de *gestión* (o las sugiere el cuaderno) |
| Cifras XBRL por API como verdad de referencia | **Tabla de estados financieros diligenciada por ti** (con borrador opcional de Yahoo Finance) |
| FinBERT directamente sobre el texto | **Traducción es→en + FinBERT**, o un modelo multilingüe, validado con el LLM |
| Embeddings `all-MiniLM-L6-v2` y consultas en inglés | Embeddings **multilingües** y consultas en español |
| Cifras `5,610.5` en millones de USD | Cifras `5.610,5` en millones, miles de millones o **billones** de COP |

### ¿Qué entregas? (sección 8 del cuaderno de la sesión 8)

1. Las secciones de **riesgos** y de **gestión** de los dos últimos informes, con el análisis de tono y
   las oraciones nuevas comentadas → secciones 1 a 3.
2. La **tabla de verificación** y un párrafo sobre los errores del LLM que encontraste → sección 5.
3. Tu propia versión de `REGLAS`, **justificada** → sección 6.3.
4. La **nota de valoración final, editada y firmada por ti**, señalando qué cambiaste del borrador de la
   IA → secciones 6.5 y 7.

Las celdas marcadas con ✍️ son las que debes escribir tú.

### 0.1 Entorno de ejecución

Igual que en las sesiones 7 y 8: pensado para **Colab**. Las partes 1 a 3 corren en CPU (más lento); las
partes con LLM (4 a 6) necesitan **GPU T4** si usas el modelo local, o una **clave de API**.

In [ ]:
# Mismas versiones fijadas de las sesiones 7 y 8 (no reinstalamos torch: Colab ya trae la build correcta para su GPU)
!pip install -q \
    "transformers==4.46.3" \
    "accelerate==1.0.1" \
    "bitsandbytes==0.44.1" \
    "sentence-transformers==3.2.1" \
    "huggingface_hub==0.25.2" \
    "scikit-learn==1.5.2" \
    "pandas==2.2.3" \
    "matplotlib==3.9.2" \
    "seaborn==0.13.2" \
    "pypdf==5.1.0" \
    "sentencepiece" \
    "sacremoses"

In [ ]:
import os
import re
import json
import random
import difflib
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import torch
from pypdf import PdfReader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Markdown

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
pd.set_option("display.max_colwidth", 160)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | dispositivo: {DEVICE}")

### 0.2 Parámetros del taller

- `EMPRESA`: nombre del emisor. Funciona con cualquier empresa **no financiera** (para bancos,
  aseguradoras y *holdings* financieros el DCF por flujo de caja libre de la firma no aplica).
- `PDFS`: un PDF **por año**, para los dos últimos ejercicios. Puede ser una ruta local (súbelo a Colab
  desde el panel de archivos) o una URL directa al PDF.
- `PAGINAS`: rango de páginas `(inicio, fin)` de cada sección, **numeradas como en el visor de PDF**
  (la primera página del archivo es la 1). Si lo dejas en `None`, el cuaderno elige páginas
  automáticamente por densidad de palabras clave (sección 1.3); es un punto de partida, no un sustituto
  de mirar la tabla de contenido.
- `TICKER_YF`: símbolo en Yahoo Finance (las acciones de la BVC suelen terminar en `.CL`; verifica el
  símbolo en la página de Yahoo). Se usa solo para el precio y para un **borrador** de las cifras.
- `FUENTE`: `"pdf"` (informes reales) o `"ficticio"` (un informe didáctico embebido, que garantiza que
  el cuaderno corra de principio a fin). Si los PDF no existen, se usa automáticamente el plan B.

**¿Dónde consigo los informes?** En la página de *relación con inversionistas* del emisor (busca
"informe periódico de fin de ejercicio", "informe de gestión" o "informe integrado") o en el Registro
Nacional de Valores y Emisores (RNVE / SIMEV) de la Superintendencia Financiera.

In [ ]:
EMPRESA = "Nombre del emisor S.A."      # ✍️ la empresa de tu proyecto integrador
TICKER_YF = None                         # ✍️ símbolo en Yahoo Finance (termina en ".CL"), o None
FUENTE = "pdf"                           # "pdf" o "ficticio"

PDFS = {                                 # ✍️ {año del ejercicio: ruta local o URL del PDF}
    2024: "informe_2024.pdf",
    2025: "informe_2025.pdf",
}
PAGINAS = {                              # ✍️ (página inicial, página final) o None = selección automática
    2024: {"riesgos": None, "gestion": None},
    2025: {"riesgos": None, "gestion": None},
}

if FUENTE == "pdf" and not all(str(p).startswith("http") or os.path.exists(str(p)) for p in PDFS.values()):
    print("⚠️  No encontré los PDF indicados en PDFS: usaremos el informe ficticio (plan B).")
    FUENTE = "ficticio"

In [ ]:
# Opcional (solo en Colab): subir los PDF desde tu computador
# from google.colab import files
# files.upload()

### 0.3 El LLM: local (Qwen) o vía API

La misma función `llamar_llm(mensajes)` de la sesión 8. Qwen 2.5 entiende bien el español, pero recuerda
que un modelo de 3B se equivoca más al extraer cifras: eso es parte de lo que vas a documentar.

In [ ]:
BACKEND_LLM = "local"                       # "local" o "api"
LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct" # modelo local (prueba "Qwen/Qwen2.5-7B-Instruct" si tienes GPU)
API_BASE_URL = "https://api.openai.com/v1"  # cambia según tu proveedor
API_MODEL = "gpt-4o-mini"                   # nombre del modelo en ese proveedor

if BACKEND_LLM == "local":
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    tokenizer_llm = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
    if tokenizer_llm.pad_token is None:
        tokenizer_llm.pad_token = tokenizer_llm.eos_token
    if DEVICE == "cuda":
        memoria_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        if memoria_gb < 10:
            cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                     bnb_4bit_quant_type="nf4")
            modelo_llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL_NAME, quantization_config=cfg,
                                                              device_map="auto")
        else:
            modelo_llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL_NAME, torch_dtype=torch.float16,
                                                              device_map="auto")
    else:
        print("⚠️  Sin GPU: el LLM local será muy lento. Considera BACKEND_LLM = 'api'.")
        modelo_llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL_NAME)
    modelo_llm.eval()
    modelo_llm.generation_config.do_sample = False
    modelo_llm.generation_config.temperature = None
    modelo_llm.generation_config.top_p = None
    modelo_llm.generation_config.top_k = None

    def llamar_llm(mensajes, max_new_tokens=400):
        prompt = tokenizer_llm.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
        entradas = tokenizer_llm(prompt, return_tensors="pt").to(modelo_llm.device)
        with torch.no_grad():
            salida = modelo_llm.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=False,
                                         pad_token_id=tokenizer_llm.pad_token_id)
        return tokenizer_llm.decode(salida[0, entradas["input_ids"].shape[1]:], skip_special_tokens=True)

else:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("LLM_API_KEY")
    except Exception:
        API_KEY = os.environ.get("LLM_API_KEY")

    def llamar_llm(mensajes, max_new_tokens=400):
        r = requests.post(
            f"{API_BASE_URL}/chat/completions",
            headers={"Authorization": f"Bearer {API_KEY}"},
            json={"model": API_MODEL, "messages": mensajes, "temperature": 0, "max_tokens": max_new_tokens},
            timeout=120,
        )
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"]

print(f"LLM listo ({BACKEND_LLM}).")

In [ ]:
def extraer_json(texto):
    # Devuelve el primer objeto JSON balanceado que aparezca en `texto`, o None.
    inicio = texto.find("{")
    while inicio != -1:
        profundidad, en_cadena, escape = 0, False, False
        for i in range(inicio, len(texto)):
            c = texto[i]
            if en_cadena:
                escape = (c == "\\") and not escape
                if c == '"' and not escape:
                    en_cadena = False
                continue
            if c == '"':
                en_cadena = True
            elif c == "{":
                profundidad += 1
            elif c == "}":
                profundidad -= 1
                if profundidad == 0:
                    try:
                        return json.loads(texto[inicio:i + 1])
                    except json.JSONDecodeError:
                        break
        inicio = texto.find("{", inicio + 1)
    return None


def normalizar_espacios(texto):
    return re.sub(r"\s+", " ", texto.replace("\xa0", " ")).strip()

<a id="sec1"></a>
## 1. Datos: el informe en PDF y los estados financieros

### 1.1 Anatomía de un informe colombiano

Los emisores inscritos en el RNVE publican cada año el **informe periódico de fin de ejercicio**
(Decreto 151 de 2021 y Circular Externa 012 de 2022 de la Superintendencia Financiera), muchas veces
integrado con el *informe de gestión* y el reporte de sostenibilidad. No tiene *Items* numerados como el
10-K, pero suele traer los mismos contenidos:

| Lo que buscamos | Equivalente en el 10-K | Cómo suele llamarse en el informe colombiano |
|---|---|---|
| `riesgos` | Item 1A – Risk Factors | "Riesgos relevantes a los que está expuesto el emisor y mecanismos de mitigación", "Gestión de riesgos", "Riesgos estratégicos" |
| `gestion` | Item 7 – MD&A | "Análisis de la administración sobre los resultados de la operación y la situación financiera", "Desempeño financiero", "Resultados del año", "Perspectivas" |
| Cifras base | Item 8 + XBRL | Estados financieros **consolidados** (anexos): resultados, situación financiera y flujos de efectivo |

Tres diferencias que vas a sentir:

- **El título y el orden cambian de empresa a empresa** (y a veces de un año a otro): por eso la sección
  se define por páginas y no por una expresión regular.
- **El PDF trae ruido**: encabezados y pies repetidos, tablas, pies de foto. Hay que limpiarlo antes de
  medir tono o comparar años.
- **No hay una API de cifras** como la de EDGAR: la verdad de referencia la construyes tú a partir de los
  estados financieros consolidados.

### 1.2 Plan B: un informe didáctico embebido

*Alimentos Cordillera S.A.* es una empresa **inventada**, con dos años de riesgos y de análisis de la
administración escritos al estilo de un informe colombiano (cifras en miles de millones de pesos, con
punto de miles y coma decimal). Tiene los mismos fenómenos del caso de la sesión 8: riesgos nuevos, un
margen que cae y una guía de crecimiento que no se cumplió.

In [ ]:
AVISO = ("[Documento ilustrativo con fines académicos: Alimentos Cordillera S.A. es una empresa ficticia "
         "y todos los nombres y cifras son inventados]. ")

REPORTE_FICTICIO = {
    2024: {
        "riesgos": AVISO + """RIESGOS RELEVANTES A LOS QUE ESTÁ EXPUESTA LA COMPAÑÍA Y MECANISMOS DE MITIGACIÓN.
La desaceleración del consumo de los hogares puede afectar nuestros resultados. Nuestros productos compiten por el gasto de los hogares colombianos. Un deterioro de las condiciones económicas, una inflación persistente o tasas de interés altas podrían reducir la demanda de alimentos empacados y afectar nuestros ingresos y nuestra rentabilidad.
Operamos en un mercado altamente competitivo. Competimos con multinacionales de alimentos y con las marcas propias de las grandes cadenas de supermercados. La presión competitiva podría obligarnos a aumentar la actividad promocional o a reducir precios, lo que disminuiría nuestro margen bruto.
Los precios de las materias primas son volátiles. El cacao, el trigo, el azúcar y los aceites representan una parte importante de nuestro costo de ventas. Aumentos significativos en sus precios internacionales podrían incrementar nuestros costos, y es posible que no logremos trasladarlos al consumidor.
La variación de la tasa de cambio afecta nuestros costos. Cerca del 40% de nuestras materias primas se compra en dólares. Una devaluación del peso colombiano encarece esas compras, aunque la compañía mitiga parcialmente este riesgo con coberturas cambiarias y con los ingresos de sus exportaciones.
Una falla en nuestros sistemas de información podría afectar el negocio. Dependemos de sistemas de información para operar nuestras plantas y nuestros centros de distribución. Un incidente de ciberseguridad podría interrumpir la operación, exponernos a litigios y afectar nuestra reputación.
Los cambios regulatorios y tributarios pueden afectar nuestra rentabilidad. La compañía está sujeta a normas sanitarias, de etiquetado y tributarias que pueden modificarse y aumentar nuestros costos de cumplimiento.""",
        "gestion": AVISO + """ANÁLISIS DE LA ADMINISTRACIÓN SOBRE LOS RESULTADOS DE LA OPERACIÓN Y LA SITUACIÓN FINANCIERA.
Alimentos Cordillera produce y comercializa galletas, chocolates y pastas en Colombia y exporta a Centroamérica y la región Andina.
Resultados de 2024 frente a 2023. Los ingresos operacionales consolidados aumentaron 9,7% y alcanzaron $5.320 miles de millones en 2024, frente a $4.850 miles de millones en 2023, impulsados por el buen desempeño del negocio de galletas y por el crecimiento de las exportaciones. Las ventas en Colombia crecieron 8,3% hasta $3.910 miles de millones, y las ventas internacionales aumentaron 13,7% hasta $1.410 miles de millones.
El margen bruto se amplió 60 puntos básicos hasta 46,1%, gracias a menores costos de fletes y a una mezcla de canales favorable. La utilidad operacional creció 9,7% y llegó a $702 miles de millones, con un margen operacional de 13,2%, igual al del año anterior. La tasa efectiva de impuestos fue de 33,0% en 2024, frente a 33,0% en 2023.
Liquidez y recursos de capital. El flujo de efectivo generado por la operación fue de $655 miles de millones. Las inversiones en propiedad, planta y equipo (capex) sumaron $240 miles de millones, destinadas principalmente a la ampliación de líneas de producción y a tecnología. El gasto por depreciaciones y amortizaciones fue de $188 miles de millones. La Asamblea decretó dividendos por $85 miles de millones. Al cierre del año la compañía tenía efectivo y equivalentes por $760 miles de millones y obligaciones financieras por $1.150 miles de millones.
Perspectivas. Para 2025 esperamos que los ingresos operacionales crezcan entre 8% y 10% y que el margen operacional se amplíe moderadamente. Prevemos inversiones de capital cercanas a $300 miles de millones, que incluyen la construcción de un nuevo centro de distribución.""",
    },
    2025: {
        "riesgos": AVISO + """RIESGOS RELEVANTES A LOS QUE ESTÁ EXPUESTA LA COMPAÑÍA Y MECANISMOS DE MITIGACIÓN.
La desaceleración del consumo de los hogares puede afectar nuestros resultados. Nuestros productos compiten por el gasto de los hogares colombianos. Un deterioro de las condiciones económicas, una inflación persistente o tasas de interés altas podrían reducir la demanda de alimentos empacados y afectar nuestros ingresos y nuestra rentabilidad.
Operamos en un mercado altamente competitivo. Competimos con multinacionales de alimentos y con las marcas propias de las grandes cadenas de supermercados. La presión competitiva podría obligarnos a aumentar la actividad promocional o a reducir precios, lo que disminuiría nuestro margen bruto.
Los impuestos saludables a los alimentos ultraprocesados pueden reducir de forma material nuestra demanda. Durante 2025 aumentó la tarifa del impuesto a los productos comestibles ultraprocesados, que grava una parte significativa de nuestro portafolio de galletas y chocolates. Este impuesto elevó los precios al consumidor y redujo los volúmenes vendidos. Nuevos incrementos de la tarifa podrían afectar de manera material y adversa nuestra rentabilidad, y es posible que no logremos reformular los productos gravados.
Los precios de las materias primas son volátiles. El cacao, el trigo, el azúcar y los aceites representan una parte importante de nuestro costo de ventas. Durante 2025 el precio internacional del cacao alcanzó máximos históricos y aumentó de forma significativa nuestros costos, y estas presiones podrían continuar.
Dependemos de un número reducido de clientes del canal moderno. Nuestro principal cliente representó aproximadamente el 14% de los ingresos en 2025. La pérdida de este cliente, o una reducción importante de sus pedidos, podría afectar de forma adversa nuestros resultados.
La variación de la tasa de cambio afecta nuestros costos. Cerca del 41% de nuestras materias primas se compra en dólares. Una devaluación del peso colombiano encarece esas compras, aunque la compañía mitiga parcialmente este riesgo con coberturas cambiarias y con los ingresos de sus exportaciones.
Una falla en nuestros sistemas de información podría afectar el negocio. Dependemos de sistemas de información para operar nuestras plantas y nuestros centros de distribución. Un incidente de ciberseguridad podría interrumpir la operación, exponernos a litigios y afectar nuestra reputación.
Los cambios regulatorios y tributarios pueden afectar nuestra rentabilidad. La compañía está sujeta a normas sanitarias, de etiquetado y tributarias que pueden modificarse y aumentar nuestros costos de cumplimiento.""",
        "gestion": AVISO + """ANÁLISIS DE LA ADMINISTRACIÓN SOBRE LOS RESULTADOS DE LA OPERACIÓN Y LA SITUACIÓN FINANCIERA.
Alimentos Cordillera produce y comercializa galletas, chocolates y pastas en Colombia y exporta a Centroamérica y la región Andina.
Resultados de 2025 frente a 2024. Los ingresos operacionales consolidados aumentaron 5,5% y alcanzaron $5.610 miles de millones en 2025, frente a $5.320 miles de millones en 2024. El crecimiento fue inferior a nuestra expectativa inicial, porque la menor demanda del canal moderno en Colombia contrarrestó el buen desempeño de las exportaciones. Las ventas en Colombia crecieron 2,6% hasta $4.010 miles de millones, mientras que las ventas internacionales aumentaron 13,5% hasta $1.600 miles de millones.
El margen bruto disminuyó 180 puntos básicos hasta 44,3%, principalmente por el mayor precio del cacao, el impuesto a los alimentos ultraprocesados y una mayor actividad promocional. La utilidad operacional disminuyó 5,7% hasta $662 miles de millones, y el margen operacional se redujo a 11,8% desde 13,2%. La tasa efectiva de impuestos fue de 33,1%, similar a la de 2024.
Liquidez y recursos de capital. El flujo de efectivo generado por la operación fue de $610 miles de millones. Las inversiones en propiedad, planta y equipo (capex) sumaron $310 miles de millones, destinadas principalmente a la construcción del nuevo centro de distribución en Yumbo. El gasto por depreciaciones y amortizaciones fue de $205 miles de millones. La Asamblea decretó dividendos por $90 miles de millones. Al cierre del año la compañía tenía efectivo y equivalentes por $820 miles de millones y obligaciones financieras por $1.150 miles de millones.
Perspectivas. Para 2026 esperamos que los ingresos operacionales crezcan entre 3% y 5% y que el margen operacional se mantenga estable o disminuya hasta 50 puntos básicos frente a 2025, por la presión de las materias primas, compensada parcialmente por iniciativas de eficiencia. Prevemos inversiones de capital cercanas a $260 miles de millones, a medida que se termina el centro de distribución.""",
    },
}

# Estados financieros del caso ficticio, en MILLONES de COP (acciones en millones)
FINANCIEROS_FICTICIOS = pd.DataFrame(
    {
        "ingresos":                 [4_850_000, 5_320_000, 5_610_000],
        "utilidad_operativa":       [640_000, 702_000, 662_000],
        "utilidad_antes_impuestos": [625_000, 690_000, 641_000],
        "impuestos":                [206_000, 228_000, 212_000],
        "dep_amort":                [172_000, 188_000, 205_000],
        "capex":                    [210_000, 240_000, 310_000],
        "caja":                     [550_000, 760_000, 820_000],
        "deuda":                    [1_150_000, 1_150_000, 1_150_000],
        "minoritarios":             [0, 0, 0],
        "acciones":                 [460, 460, 460],
    },
    index=pd.Index([2023, 2024, 2025], name="anio"),
)
PRECIO_FICTICIO = 7_200.0   # precio de mercado inventado, COP por acción

### 1.3 Plan A: leer el PDF y ubicar las secciones

`leer_paginas_pdf` devuelve **una cadena de texto por página** y hace dos limpiezas típicas de PDF:

- une las palabras partidas con guion al final de la línea (`opera-` / `ción` → `operación`);
- elimina **encabezados y pies de página**: líneas que se repiten (ignorando los números) en más del 30 %
  de las páginas. Si no se quitan, inflan artificialmente el parecido entre años.

> **PDF escaneado.** Si el texto sale casi vacío, el PDF es una imagen: necesitas la versión digital del
> informe o pasarlo antes por un OCR (por ejemplo `ocrmypdf`).

In [ ]:
def obtener_pdf(fuente, destino):
    # Si `fuente` es una URL la descarga a `destino`; si es una ruta local la devuelve tal cual.
    if str(fuente).startswith("http"):
        r = requests.get(fuente, headers={"User-Agent": "Mozilla/5.0"}, timeout=180)
        r.raise_for_status()
        with open(destino, "wb") as f:
            f.write(r.content)
        return destino
    return fuente


def _clave_linea(linea):
    return re.sub(r"\d+", "#", linea.strip().lower())


def leer_paginas_pdf(ruta):
    crudas = [(p.extract_text() or "") for p in PdfReader(ruta).pages]
    crudas = [re.sub(r"([a-záéíóúñ])-\s*\n\s*([a-záéíóúñ])", r"\1\2", t) for t in crudas]
    conteo = Counter(k for t in crudas for k in {_clave_linea(l) for l in t.splitlines() if l.strip()})
    repetidas = {k for k, n in conteo.items() if n > max(3, 0.3 * len(crudas))}
    paginas = [normalizar_espacios(" ".join(l for l in t.splitlines() if _clave_linea(l) not in repetidas))
               for t in crudas]
    if sum(len(p.split()) for p in paginas) < 50 * len(paginas):
        print(f"⚠️  {ruta}: muy poco texto por página. ¿Es un PDF escaneado? Necesitarás OCR.")
    return paginas

**Ubicar las secciones.** `mapa_paginas` calcula, para cada página, cuántas palabras clave de *riesgos* y
de *gestión* aparecen por cada 1 000 palabras, y qué proporción de sus caracteres son dígitos (las páginas
que son casi solo tablas tienen muchos). Úsalo junto con la tabla de contenido del informe para llenar
`PAGINAS`.

Si dejas una sección en `None`, se toman las `MAX_PAGINAS_AUTO` páginas de prosa con mayor densidad. Es
un atajo razonable para empezar, pero tiene sesgos conocidos (por ejemplo, las notas a los estados
financieros sobre "gestión del riesgo financiero" puntúan alto en `riesgos`): **en la entrega, las páginas
deben ser las que tú verificaste**.

In [ ]:
PALABRAS_SECCION = {
    "riesgos": r"riesgo|mitigaci[oó]n|incertidumbre|contingencia|volatilidad|exposici[oó]n|adverso",
    "gestion": r"ingresos|ventas|ebitda|margen|utilidad|crecimiento|perspectivas|inversiones|capex|endeudamiento",
}
MAX_PAGINAS_AUTO = 15


def mapa_paginas(paginas):
    filas = []
    for i, texto in enumerate(paginas, start=1):
        n = len(texto.split())
        fila = {"pagina": i, "palabras": n,
                "pct_digitos": sum(c.isdigit() for c in texto) / max(len(texto), 1)}
        for seccion, patron in PALABRAS_SECCION.items():
            fila[seccion] = 1000 * len(re.findall(patron, texto, flags=re.I)) / max(n, 1)
        filas.append(fila)
    return pd.DataFrame(filas).set_index("pagina")


def seleccionar_paginas(paginas, seccion, rango=None):
    if rango:
        return list(range(rango[0], min(rango[1], len(paginas)) + 1))
    m = mapa_paginas(paginas)
    prosa = m[(m["palabras"] >= 120) & (m["pct_digitos"] < 0.12)]
    return sorted(prosa[seccion].nlargest(MAX_PAGINAS_AUTO).index)

In [ ]:
TEXTOS = {}            # {año: {"riesgos": texto, "gestion": texto}}
PAGINAS_PDF = {}       # {año: [texto de cada página]}
PAGINAS_USADAS = {}    # {año: {sección: [páginas]}}

if FUENTE == "pdf":
    try:
        for anio, fuente in sorted(PDFS.items()):
            PAGINAS_PDF[anio] = leer_paginas_pdf(obtener_pdf(fuente, f"informe_{anio}.pdf"))
            PAGINAS_USADAS[anio], TEXTOS[anio] = {}, {}
            for seccion in PALABRAS_SECCION:
                pags = seleccionar_paginas(PAGINAS_PDF[anio], seccion, PAGINAS.get(anio, {}).get(seccion))
                PAGINAS_USADAS[anio][seccion] = pags
                TEXTOS[anio][seccion] = " ".join(PAGINAS_PDF[anio][p - 1] for p in pags)
                modo = "manual" if PAGINAS.get(anio, {}).get(seccion) else "automático"
                print(f"{anio} | {seccion:8s} | {len(TEXTOS[anio][seccion].split()):>7,} palabras | "
                      f"páginas ({modo}): {pags}")
        if len(TEXTOS) < 2:
            raise ValueError("Se necesitan los informes de dos años para comparar")
        if any(len(s.split()) < 300 for t in TEXTOS.values() for s in t.values()):
            raise ValueError("Alguna sección quedó casi vacía: revisa PAGINAS o el PDF")
    except Exception as error:
        print(f"Plan A falló → usamos el informe ficticio.\n  {type(error).__name__}: {str(error)[:200]}")
        FUENTE = "ficticio"

if FUENTE == "ficticio":
    EMPRESA = "Alimentos Cordillera S.A. (ficticia)"
    TICKER_YF = None
    TEXTOS = {anio: {k: normalizar_espacios(v) for k, v in secc.items()} for anio, secc in REPORTE_FICTICIO.items()}

ANIOS = sorted(TEXTOS)
ANIO_ACTUAL, ANIO_PREVIO = ANIOS[-1], ANIOS[0]
print(f"\nEmpresa: {EMPRESA} | años de texto: {ANIOS}")

In [ ]:
# Ayuda para llenar PAGINAS: las páginas con mayor densidad de cada tema en el informe más reciente
if FUENTE == "pdf":
    m = mapa_paginas(PAGINAS_PDF[ANIO_ACTUAL]).round(3)
    for seccion in PALABRAS_SECCION:
        display(Markdown(f"**Páginas con más vocabulario de `{seccion}` ({ANIO_ACTUAL}):**"))
        display(m.nlargest(8, seccion).sort_index().T)
    # Para leer una página concreta: print(PAGINAS_PDF[ANIO_ACTUAL][45 - 1][:1500])

### 1.4 Estados financieros: la verdad de referencia la construyes tú

Sin XBRL por API, las cifras salen de los **estados financieros consolidados** del informe. Diligencia
`FINANCIEROS_MANUALES` con los **tres últimos años**, todo en **millones de COP** (acciones en millones):

| Variable | Dónde está | Ojo con |
|---|---|---|
| `ingresos` | Estado de resultados: ingresos de actividades ordinarias | Total consolidado, no un segmento |
| `utilidad_operativa` | Estado de resultados: utilidad operacional / resultado de actividades de operación | No es el EBITDA |
| `utilidad_antes_impuestos`, `impuestos` | Estado de resultados | Impuesto corriente + diferido |
| `dep_amort` | Flujo de efectivo (ajustes) o notas | Incluye amortización de intangibles y derechos de uso |
| `capex` | Flujo de efectivo: adquisición de propiedad, planta y equipo (e intangibles) | Con signo **positivo** |
| `caja` | Estado de situación financiera: efectivo y equivalentes | Saldo al cierre |
| `deuda` | Obligaciones financieras + bonos (corriente y no corriente) | Decide y documenta si incluyes arrendamientos (NIIF 16) |
| `minoritarios` | Patrimonio: participaciones no controladoras | Clave en *holdings*; 0 si no hay |
| `acciones` | Notas de patrimonio: acciones en circulación (ordinarias + preferenciales) | En millones |

> **Unidades.** Los informes colombianos expresan las cifras en *millones*, *miles de millones* o
> *billones* de pesos (un billón = un millón de millones = $10^{12}$). Pasa todo a **millones de COP**:
> 5,61 billones = 5.610 miles de millones = 5.610.000 millones.

**Borrador opcional.** Si definiste `TICKER_YF`, `financieros_yfinance` trae un borrador desde Yahoo
Finance. Sirve para ahorrar digitación, **no** como verdad de referencia: Yahoo reclasifica partidas y a
veces reporta en otra moneda. Lo que escribas en `FINANCIEROS_MANUALES` siempre prevalece.

In [ ]:
VARIABLES = ["ingresos", "utilidad_operativa", "utilidad_antes_impuestos", "impuestos", "dep_amort",
             "capex", "caja", "deuda", "minoritarios", "acciones"]

# ✍️ Millones de COP (acciones en millones). Ejemplo: FINANCIEROS_MANUALES.loc[2025, "ingresos"] = 5_610_000
FINANCIEROS_MANUALES = pd.DataFrame(np.nan, index=pd.Index([2023, 2024, 2025], name="anio"), columns=VARIABLES)

CONCEPTOS_YF = {
    "ingresos": ("income_stmt", ["Total Revenue", "Operating Revenue"]),
    "utilidad_operativa": ("income_stmt", ["Operating Income"]),
    "utilidad_antes_impuestos": ("income_stmt", ["Pretax Income"]),
    "impuestos": ("income_stmt", ["Tax Provision"]),
    "dep_amort": ("cashflow", ["Depreciation And Amortization", "Depreciation Amortization Depletion"]),
    "capex": ("cashflow", ["Capital Expenditure"]),
    "caja": ("balance_sheet", ["Cash And Cash Equivalents"]),
    "deuda": ("balance_sheet", ["Total Debt"]),
    "minoritarios": ("balance_sheet", ["Minority Interest"]),
    "acciones": ("income_stmt", ["Diluted Average Shares", "Basic Average Shares"]),
}


def financieros_yfinance(ticker):
    # Borrador de los estados financieros anuales desde Yahoo Finance, en millones. Hay que verificarlo.
    import yfinance as yf
    t = yf.Ticker(ticker)
    estados = {nombre: getattr(t, nombre) for nombre in ["income_stmt", "cashflow", "balance_sheet"]}
    columnas = {}
    for var, (estado, conceptos) in CONCEPTOS_YF.items():
        for concepto in conceptos:
            if concepto in estados[estado].index:
                serie = estados[estado].loc[concepto].dropna()
                serie.index = pd.to_datetime(serie.index).year
                columnas[var] = serie.abs() / 1e6
                break
    moneda = (t.info or {}).get("financialCurrency")
    print(f"Borrador de Yahoo Finance para {ticker} | moneda de los estados financieros: {moneda}")
    if moneda not in (None, "COP"):
        print("⚠️  Las cifras NO están en COP: no las mezcles con las del informe sin convertirlas.")
    return pd.DataFrame(columnas).rename_axis("anio").sort_index()


if FUENTE == "ficticio":
    FINANCIEROS = FINANCIEROS_FICTICIOS.copy()
else:
    FINANCIEROS = FINANCIEROS_MANUALES.copy()
    if TICKER_YF:
        try:
            FINANCIEROS = FINANCIEROS.combine_first(financieros_yfinance(TICKER_YF)).reindex(columns=VARIABLES)
        except Exception as error:
            print(f"No pude traer el borrador de Yahoo Finance ({type(error).__name__}: {str(error)[:120]}).")
    FINANCIEROS = FINANCIEROS.loc[FINANCIEROS.index <= ANIO_ACTUAL].tail(3)
    FINANCIEROS["minoritarios"] = FINANCIEROS["minoritarios"].fillna(0)

FINANCIEROS

In [ ]:
faltantes = FINANCIEROS.isna()
if faltantes.values.any():
    print("⚠️  Variables sin dato (complétalas en FINANCIEROS_MANUALES antes de las secciones 5 y 6):")
    print(faltantes.stack()[lambda s: s].index.tolist())
else:
    print("✅ Todas las variables tienen dato para los años analizados.")
if ANIO_ACTUAL not in FINANCIEROS.index:
    print(f"⚠️  FINANCIEROS no tiene el año {ANIO_ACTUAL}, que es el del informe más reciente.")

**Precio de mercado**, en COP por acción. Si `yfinance` falla o no definiste `TICKER_YF`, escríbelo a mano
(precio de cierre de la BVC en la fecha de tu análisis).

In [ ]:
PRECIO_MERCADO = None      # ✍️ COP por acción, si no usas yfinance
if FUENTE == "ficticio":
    PRECIO_MERCADO = PRECIO_FICTICIO
elif TICKER_YF and PRECIO_MERCADO is None:
    try:
        import yfinance as yf
        PRECIO_MERCADO = float(yf.Ticker(TICKER_YF).history(period="5d")["Close"].iloc[-1])
    except Exception as error:
        print(f"No pude descargar el precio ({error}). Escríbelo a mano en PRECIO_MERCADO.")
print(f"Precio de mercado: {PRECIO_MERCADO}")

<a id="sec2"></a>
## 2. Segmentación: oraciones y *chunks* en español

Dos ajustes frente a la sesión 8:

- **Abreviaturas colombianas** (`S.A.`, `S.A.S.`, `E.S.P.`, `Ltda.`, `No.`, `Art.`) que no deben cortar
  una oración, y un **filtro de prosa**: descartamos fragmentos que son filas de tablas o títulos sueltos
  (muy cortos, muy largos o con menos de 60 % de letras).
- **Chunks de ~110 palabras**. El modelo de embeddings multilingüe que usaremos trunca por defecto a 128
  *tokens*; lo ampliamos a 256, que en español equivalen a unas 120–150 palabras.

In [ ]:
ABREVIATURAS = r"\b(S\.A\.S|S\.A|E\.S\.P|Ltda|Cía|No|Núm|núm|Art|art|aprox|Dr|Dra|Sr|Sra|vs|Inc|Corp|EE|UU)\."


def es_prosa(oracion, min_palabras=6, max_palabras=120):
    n = len(oracion.split())
    letras = sum(c.isalpha() for c in oracion) / max(len(oracion), 1)
    return min_palabras <= n <= max_palabras and letras >= 0.6


def dividir_oraciones(texto):
    protegido = re.sub(ABREVIATURAS, lambda m: m.group(0).replace(".", "§"), normalizar_espacios(texto))
    partes = re.split(r"(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÑ¿¡\"(\[])", protegido)
    return [p.replace("§", ".").strip() for p in partes if es_prosa(p)]


def crear_chunks(oraciones, max_palabras=110, solape=1):
    chunks, actual = [], []
    for o in oraciones:
        if actual and sum(len(x.split()) for x in actual) + len(o.split()) > max_palabras:
            chunks.append(" ".join(actual))
            actual = actual[-solape:] if solape else []
        actual.append(o)
    if actual:
        chunks.append(" ".join(actual))
    return chunks


filas = []
for anio in ANIOS:
    for seccion, texto in TEXTOS[anio].items():
        for o in dividir_oraciones(texto):
            filas.append({"anio": anio, "seccion": seccion, "oracion": o})
df_oraciones = pd.DataFrame(filas).drop_duplicates().reset_index(drop=True)

print(df_oraciones.groupby(["anio", "seccion"]).size().rename("n_oraciones"))
df_oraciones.sample(min(5, len(df_oraciones)), random_state=SEED)

In [ ]:
# Chunks del informe más reciente: la base de conocimiento del RAG
filas = []
for seccion in ["riesgos", "gestion"]:
    oraciones = df_oraciones.query("anio == @ANIO_ACTUAL and seccion == @seccion")["oracion"].tolist()
    for texto in crear_chunks(oraciones):
        filas.append({"seccion": seccion, "texto": texto})
df_chunks = pd.DataFrame(filas)
df_chunks.insert(0, "id", [f"C{i}" for i in range(len(df_chunks))])

print(f"{len(df_chunks)} chunks | palabras por chunk: "
      f"{df_chunks['texto'].str.split().str.len().describe()[['mean', 'max']].round(0).to_dict()}")
df_chunks.head(3)

> **Revisa la muestra de oraciones.** Si ves restos de tablas, índices o pies de foto, ajusta `PAGINAS`
> (o los umbrales de `es_prosa`) antes de seguir: todo lo que viene después hereda este ruido.

<a id="sec3"></a>
## 3. ¿Cambió el tono? ¿Qué riesgos son nuevos?

### 3.1 Tono por sección: ¿qué usamos si FinBERT no habla español?

FinBERT fue entrenado con textos financieros **en inglés**; aplicado directamente al español clasifica casi
al azar. Hay tres salidas razonables, y cada una falla de una manera distinta:

| Método | Ventaja | Limitación |
|---|---|---|
| `"traduccion_finbert"`: traducir es→en (`Helsinki-NLP/opus-mt-es-en`) y aplicar FinBERT | Conserva un clasificador de **dominio financiero** | El traductor puede distorsionar jerga contable; es el más lento |
| `"xlmr"`: modelo multilingüe (`cardiffnlp/twitter-xlm-roberta-base-sentiment`) | Directo sobre el español, rápido | Entrenado con **tuits**, no con lenguaje financiero |
| LLM *zero-shot* (como en la sesión 7) | Entiende contexto financiero en español | Lento y costoso para miles de oraciones |

Usamos el primero como método principal (puedes cambiar `METODO_TONO`) y el LLM **como auditor**: clasifica
una muestra y medimos el acuerdo. El indicador es el mismo **tono neto** de la sesión 8:

$$\text{tono neto} = \frac{\#\text{positivas} - \#\text{negativas}}{\#\text{oraciones}}$$

La sección de riesgos **siempre** es negativa por diseño; lo informativo es **su cambio entre años**.

In [ ]:
from transformers import pipeline

METODO_TONO = "traduccion_finbert"     # "traduccion_finbert" o "xlmr"
_dispositivo = 0 if DEVICE == "cuda" else -1
_ETIQUETAS = {"label_0": "negative", "label_1": "neutral", "label_2": "positive"}

if METODO_TONO == "traduccion_finbert":
    traductor = pipeline("translation", model="Helsinki-NLP/opus-mt-es-en", device=_dispositivo)
    clasificador_tono = pipeline("text-classification", model="ProsusAI/finbert", device=_dispositivo)
else:
    traductor = None
    clasificador_tono = pipeline("text-classification", model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
                                 device=_dispositivo)


def clasificar_tono(oraciones):
    # Devuelve (etiquetas, confianzas, texto que realmente leyó el clasificador).
    textos = list(oraciones)
    if traductor is not None:
        textos = [t["translation_text"] for t in traductor(textos, batch_size=16, truncation=True, max_length=256)]
    salidas = clasificador_tono(textos, batch_size=32, truncation=True)
    etiquetas = [_ETIQUETAS.get(s["label"].lower(), s["label"].lower()) for s in salidas]
    return etiquetas, [s["score"] for s in salidas], textos

In [ ]:
df_oraciones["sentimiento"], df_oraciones["confianza"], df_oraciones["texto_clasificado"] = clasificar_tono(
    df_oraciones["oracion"].tolist())

tono = (
    df_oraciones.assign(pos=df_oraciones["sentimiento"].eq("positive"),
                        neg=df_oraciones["sentimiento"].eq("negative"))
    .groupby(["seccion", "anio"])
    .agg(n=("oracion", "size"), pct_positivas=("pos", "mean"), pct_negativas=("neg", "mean"))
)
tono["tono_neto"] = tono["pct_positivas"] - tono["pct_negativas"]
display(tono.round(3))

ax = tono["tono_neto"].unstack("anio").plot.bar(figsize=(6, 4), rot=0)
ax.axhline(0, color="black", lw=0.8)
ax.set_title(f"Tono neto por sección — {EMPRESA}")
ax.set_ylabel("tono neto")
plt.show()

In [ ]:
# Las oraciones más negativas de la sección de gestión más reciente (y lo que leyó el clasificador)
(df_oraciones.query("anio == @ANIO_ACTUAL and seccion == 'gestion' and sentimiento == 'negative'")
 .sort_values("confianza", ascending=False)[["oracion", "texto_clasificado", "confianza"]].head(5))

**Auditoría con el LLM.** Tomamos una muestra de oraciones del informe más reciente, le pedimos al LLM la
misma clasificación y calculamos el porcentaje de acuerdo. Un acuerdo bajo no dice cuál de los dos se
equivoca: dice que **debes leer los desacuerdos** antes de confiar en el indicador.

In [ ]:
PROMPT_TONO = (
    "Eres un analista financiero. Clasifica el tono de la oración, tomada del informe anual de una empresa, "
    "desde el punto de vista de un inversionista: positive (buena noticia para la empresa), negative (mala "
    "noticia o riesgo) o neutral (descriptiva).\n"
    'Responde SOLO con un objeto JSON válido: {"sentimiento": "positive" | "negative" | "neutral"}'
)
N_AUDITORIA = 20

muestra = (df_oraciones.query("anio == @ANIO_ACTUAL")
           .sample(min(N_AUDITORIA, int((df_oraciones["anio"] == ANIO_ACTUAL).sum())), random_state=SEED).copy())
muestra["sentimiento_llm"] = [
    (extraer_json(llamar_llm([{"role": "system", "content": PROMPT_TONO},
                              {"role": "user", "content": o}], max_new_tokens=30)) or {}).get("sentimiento")
    for o in muestra["oracion"]
]
ACUERDO_TONO = float((muestra["sentimiento"] == muestra["sentimiento_llm"]).mean())
print(f"Acuerdo entre {METODO_TONO} y el LLM en {len(muestra)} oraciones: {ACUERDO_TONO:.0%}")
muestra.loc[muestra["sentimiento"] != muestra["sentimiento_llm"],
            ["seccion", "oracion", "sentimiento", "sentimiento_llm"]]

### 3.2 *Lazy Prices* a la colombiana: el valor del cambio

La idea de Cohen, Malloy y Nguyen (2020) se traslada tal cual: lo que la empresa **agrega** a su sección de
riesgos dice más que lo que repite. Dos ajustes técnicos:

- TF-IDF con **palabras vacías en español** (scikit-learn solo trae las del inglés).
- Embeddings **multilingües** (`paraphrase-multilingual-MiniLM-L12-v2`) para buscar, para cada oración de
  este año, la más parecida del año anterior. Si ni esa se le acerca (similitud < `UMBRAL_NOVEDAD`), la
  oración es **nueva**.

> **Advertencia propia de los PDF.** Un informe puede cambiar de diagramación o de estructura de un año a
> otro (por ejemplo, al pasar de "informe de gestión" a "informe integrado"), y eso baja la similitud sin
> que el riesgo haya cambiado. Compara **las mismas secciones** en ambos años y lee las oraciones nuevas
> antes de interpretarlas.

In [ ]:
from sentence_transformers import SentenceTransformer

modelo_embeddings = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=DEVICE)
modelo_embeddings.max_seq_length = 256      # por defecto trunca a 128 tokens

In [ ]:
STOPWORDS_ES = (
    "a al algo algunas algunos ante antes como con contra cual cuando de del desde donde durante e el ella ellas "
    "ellos en entre era es esa esas ese eso esos esta estas este esto estos fue fueron ha han hasta hay la las le "
    "les lo los más me mi muy ni no nos nuestra nuestras nuestro nuestros o os otra otras otro otros para pero poco "
    "por porque que quien se sea ser si sí sin sobre son su sus también tanto te tiene tienen todo todos un una "
    "uno unos y ya"
).split()


def similitud_tfidf(texto_a, texto_b):
    m = TfidfVectorizer(stop_words=STOPWORDS_ES).fit_transform([texto_a, texto_b])
    return float(cosine_similarity(m[0], m[1])[0, 0])


similitudes = pd.Series(
    {s: similitud_tfidf(TEXTOS[ANIO_PREVIO][s], TEXTOS[ANIO_ACTUAL][s]) for s in ["riesgos", "gestion"]},
    name=f"similitud {ANIO_PREVIO}→{ANIO_ACTUAL}",
)
similitudes.round(3)

In [ ]:
UMBRAL_NOVEDAD = 0.75   # parámetro de juicio: súbelo y aparecerán más oraciones "nuevas"


def comparar_oraciones(seccion):
    act = df_oraciones.query("anio == @ANIO_ACTUAL and seccion == @seccion").reset_index(drop=True)
    prev = df_oraciones.query("anio == @ANIO_PREVIO and seccion == @seccion").reset_index(drop=True)
    e_act = modelo_embeddings.encode(act["oracion"].tolist(), normalize_embeddings=True)
    e_prev = modelo_embeddings.encode(prev["oracion"].tolist(), normalize_embeddings=True)
    sim = e_act @ e_prev.T
    act["similitud_max"] = sim.max(axis=1)
    prev["similitud_max"] = sim.max(axis=0)
    nuevas = act[act["similitud_max"] < UMBRAL_NOVEDAD]
    eliminadas = prev[prev["similitud_max"] < UMBRAL_NOVEDAD]
    return act, nuevas, eliminadas


act_riesgos, nuevas_riesgos, eliminadas_riesgos = comparar_oraciones("riesgos")
PROP_NUEVAS_RIESGOS = len(nuevas_riesgos) / len(act_riesgos)
print(f"Riesgos {ANIO_ACTUAL}: {len(nuevas_riesgos)} de {len(act_riesgos)} oraciones son nuevas "
      f"({PROP_NUEVAS_RIESGOS:.0%}); {len(eliminadas_riesgos)} oraciones de {ANIO_PREVIO} desaparecieron.\n")
display(Markdown("**Oraciones nuevas en la sección de riesgos:**"))
display(nuevas_riesgos[["oracion", "sentimiento", "similitud_max"]].sort_values("similitud_max").head(15))
display(Markdown(f"**Oraciones de {ANIO_PREVIO} que desaparecieron:**"))
display(eliminadas_riesgos[["oracion", "similitud_max"]].sort_values("similitud_max").head(10))

### ✍️ Entregable 1 — Tono y oraciones nuevas, comentados

*Reemplaza este texto con tu análisis (10 a 15 líneas):*

- **Secciones usadas**: páginas de `riesgos` y `gestion` en cada informe, y cómo las identificaste.
- **Tono**: ¿cambió el tono neto entre años en cada sección? ¿Es coherente con los resultados financieros?
  ¿Qué acuerdo hubo con el LLM y qué viste en los desacuerdos?
- **Oraciones nuevas**: clasifica las principales en (a) riesgo genuinamente nuevo, (b) riesgo existente
  que se agravó, (c) solo cambio de redacción o de formato. ¿Cuáles tienen consecuencias para la valoración?
- **Oraciones eliminadas**: ¿algún riesgo dejó de mencionarse? ¿Por qué crees?

<a id="sec4"></a>
## 4. RAG: preguntarle al informe (y exigir citas)

El sistema es el de la sesión 8: indexar los chunks, recuperar los *k* más parecidos a la consulta,
dárselos al LLM con su id `[C#]` y verificar en Python las citas y los números. Como el modelo de
embeddings ahora es multilingüe, **las consultas de búsqueda van en español**, con el vocabulario del
informe ("ingresos operacionales", "utilidad operacional", "inversiones de capital").

In [ ]:
E_CHUNKS = modelo_embeddings.encode(df_chunks["texto"].tolist(), normalize_embeddings=True)


def recuperar(consulta, k=4, seccion=None):
    q = modelo_embeddings.encode([consulta], normalize_embeddings=True)[0]
    df = df_chunks.assign(score=E_CHUNKS @ q)
    if seccion:
        df = df[df["seccion"] == seccion]
    return df.sort_values("score", ascending=False).head(k)


recuperar("crecimiento esperado de los ingresos para el próximo año, perspectivas", k=3)[["id", "seccion", "score", "texto"]]

In [ ]:
PROMPT_RAG = (
    "Eres un analista financiero. Responde ÚNICAMENTE con base en los fragmentos del informe anual que "
    "se te entregan, cada uno identificado como [C#]. No uses conocimiento externo. "
    "Si la respuesta no está en los fragmentos, dilo y marca encontrado como false.\n"
    "Responde SOLO con un objeto JSON válido, sin texto adicional, con estas claves:\n"
    '  "respuesta": respuesta breve en español (máximo 3 frases)\n'
    '  "citas": lista con los ids de los fragmentos usados, por ejemplo ["C3", "C5"]\n'
    '  "encontrado": true o false'
)


def formatear_contexto(df_ctx):
    return "\n\n".join(f"[{r.id}] {r.texto}" for r in df_ctx.itertuples())


def _candidatos_numero(token):
    # '5.610' → {5.61, 5610}; '5,5' → {55, 5.5}: cubre el formato español y el inglés.
    token = token.strip(".,")
    candidatos = set()
    for s in (token.replace(",", ""), token.replace(".", "").replace(",", ".")):
        try:
            candidatos.add(round(float(s), 4))
        except ValueError:
            pass
    return candidatos


def numeros_en(texto):
    candidatos = set()
    for token in re.findall(r"(?<![A-Za-z\d])\d[\d.,]*", texto):
        candidatos |= _candidatos_numero(token)
    return candidatos


def numeros_no_soportados(texto_respuesta, texto_fuente):
    fuente = numeros_en(texto_fuente)
    return [t.strip(".,") for t in re.findall(r"(?<![A-Za-z\d])\d[\d.,]*", texto_respuesta)
            if not (_candidatos_numero(t) & fuente)]


def preguntar_reporte(pregunta, consulta, k=4, seccion=None):
    ctx = recuperar(consulta, k=k, seccion=seccion)
    mensajes = [{"role": "system", "content": PROMPT_RAG},
                {"role": "user", "content": f"Fragmentos:\n{formatear_contexto(ctx)}\n\nPregunta: {pregunta}"}]
    datos = extraer_json(llamar_llm(mensajes, max_new_tokens=300)) or {}
    citas = [c for c in datos.get("citas", []) if c in set(ctx["id"])]
    texto_citado = " ".join(ctx.loc[ctx["id"].isin(citas), "texto"]) if citas else ""
    respuesta = str(datos.get("respuesta", ""))
    return {
        "pregunta": pregunta,
        "respuesta": respuesta,
        "citas": citas,
        "encontrado": datos.get("encontrado"),
        "citas_invalidas": [c for c in datos.get("citas", []) if c not in set(ctx["id"])],
        "numeros_sin_soporte": numeros_no_soportados(respuesta, texto_citado),
        "json_valido": bool(datos),
    }

In [ ]:
PREGUNTAS = [   # (pregunta al LLM, consulta de búsqueda)  ✍️ agrega las propias de tu empresa
    ("¿Qué crecimiento de ingresos espera la administración para el próximo año?",
     "perspectivas: crecimiento esperado de los ingresos para el próximo año"),
    ("¿Qué explica el cambio en los márgenes del último año?",
     "el margen bruto o el margen EBITDA aumentó o disminuyó debido a"),
    ("¿Cuánto espera invertir la empresa (capex) el próximo año?",
     "inversiones de capital previstas para el próximo año, plan de inversiones"),
    ("¿Cuáles son los principales riesgos para la rentabilidad?",
     "riesgos que podrían afectar de forma adversa la rentabilidad y los márgenes"),
    # Pregunta trampa: la respuesta NO está en el informe. Un buen sistema debe decir que no la encontró.
    ("¿Cuál es el precio objetivo de la acción según la administración?",
     "precio objetivo de la acción"),
]

df_rag = pd.DataFrame([preguntar_reporte(p, c) for p, c in PREGUNTAS])
df_rag

Revisa `citas_invalidas` (ids inventados), `numeros_sin_soporte` (cifras que no están en los fragmentos
citados) y la **pregunta trampa**. En informes colombianos es frecuente que **no exista guía numérica**
de crecimiento: en ese caso la respuesta correcta es `encontrado = false`, no un número.

<a id="sec5"></a>
## 5. Extracción estructurada… y su verificación

Las tres reglas de la sesión 8 (un campo por llamada, valor + cita literal + unidad, verificación en
Python) y dos cuidados nuevos, propios de los informes en español:

1. **Formato numérico.** `5.610` es cinco mil seiscientos diez y `5,5` es cinco coma cinco. Si el LLM
   devuelve el valor como número JSON, `5.610` se convierte en 5,61 sin avisar. Por eso le pedimos el
   valor como **texto, tal como aparece**, y lo interpreta `parsear_numero` en Python.
2. **Unidades.** *Millones*, *miles de millones* y *billones* conviven en el mismo informe. El LLM solo
   reporta la unidad; la conversión a millones de COP la hace `FACTOR_UNIDAD`.

> `parsear_numero` asume la convención habitual: un separador seguido de **exactamente tres dígitos** es
> de miles (`5.610`, `1,234`); en otro caso es decimal (`5,5`, `44.3`). Si tu informe usa tres decimales,
> ajústala.

In [ ]:
def parsear_numero(valor):
    # '5.610' → 5610 | '5,5' → 5.5 | '1.234,5' → 1234.5 | '1,234.5' → 1234.5 | no numérico → None
    s = re.sub(r"[^\d.,\-]", "", str(valor)).strip(".,")
    if not re.search(r"\d", s):
        return None
    if "." in s and "," in s:
        decimal = "," if s.rfind(",") > s.rfind(".") else "."
        s = s.replace("." if decimal == "," else ",", "").replace(decimal, ".")
    else:
        for sep in ".,":
            if sep in s:
                partes = s.split(sep)
                s = s.replace(sep, "") if len(partes) > 2 or len(partes[1]) == 3 else s.replace(sep, ".")
    try:
        return float(s)
    except ValueError:
        return None


pd.Series({t: parsear_numero(t) for t in ["5.610", "5,5", "1.234,5", "1,234.5", "$ 133.456.789", "44,3%", "n.d."]})

In [ ]:
CAMPOS = {   # campo: (consulta de búsqueda, descripción para el LLM)
    "ingresos": ("ingresos operacionales consolidados del año, ingresos de actividades ordinarias, ventas totales",
                 "los ingresos operacionales TOTALES consolidados de la empresa, no los de un segmento o país"),
    "utilidad_operativa": ("utilidad operacional del año, margen operacional",
                           "la utilidad operacional total (no el EBITDA)"),
    "dep_amort": ("gasto por depreciaciones y amortizaciones",
                  "el gasto de depreciaciones y amortizaciones"),
    "capex": ("inversiones en propiedad, planta y equipo, capex ejecutado en el año",
              "las inversiones de capital (capex) EJECUTADAS en el año, no las previstas para el año siguiente"),
    "tasa_impuestos": ("tasa efectiva de impuestos",
                       "la tasa efectiva de impuesto de renta, en porcentaje"),
}

# Todo se lleva a millones de COP (los porcentajes quedan igual)
FACTOR_UNIDAD = {"millones": 1, "miles de millones": 1_000, "billones": 1_000_000, "miles": 0.001, "porcentaje": 1}

PROMPT_EXTRACCION = (
    "Eres un analista que extrae datos de informes financieros colombianos. Usa SOLO los fragmentos "
    "entregados. No calcules, no estimes y no conviertas unidades.\n"
    "Responde SOLO con un objeto JSON válido con estas claves:\n"
    '  "valor": el número copiado como TEXTO entre comillas, con los mismos puntos y comas del fragmento '
    '(por ejemplo "5.610" o "44,3"), o null si no aparece\n'
    '  "unidad": una de "millones", "miles de millones", "billones", "miles" o "porcentaje", tal como la '
    "expresa el fragmento (un billón es un millón de millones)\n"
    '  "cita": la oración del fragmento que contiene el dato, copiada literalmente'
)


def cita_existe(cita, texto_fuente, umbral=0.9):
    # True si la cita aparece literalmente (o casi: ratio ≥ umbral) en alguna oración de la fuente.
    cita = normalizar_espacios(str(cita)).lower()
    if len(cita) < 15:
        return False
    if cita in normalizar_espacios(texto_fuente).lower():
        return True
    return max((difflib.SequenceMatcher(None, cita, o.lower()).ratio()
                for o in dividir_oraciones(texto_fuente)), default=0) >= umbral


def extraer_campo(campo, anio):
    consulta, descripcion = CAMPOS[campo]
    ctx = recuperar(consulta, k=4, seccion="gestion")
    pregunta = f"Extrae {descripcion} correspondiente al año {anio}."
    mensajes = [{"role": "system", "content": PROMPT_EXTRACCION},
                {"role": "user", "content": f"Fragmentos:\n{formatear_contexto(ctx)}\n\n{pregunta}"}]
    datos = extraer_json(llamar_llm(mensajes, max_new_tokens=200)) or {}
    valor = parsear_numero(datos.get("valor"))
    if valor is not None:
        valor *= FACTOR_UNIDAD.get(str(datos.get("unidad")).lower(), 1)
    return {"campo": campo, "valor_texto": datos.get("valor"), "unidad_llm": datos.get("unidad"),
            "valor_llm": valor, "cita": datos.get("cita", ""),
            "cita_verificada": cita_existe(datos.get("cita", ""), TEXTOS[ANIO_ACTUAL]["gestion"])}


df_extraccion = pd.DataFrame([extraer_campo(c, ANIO_ACTUAL) for c in CAMPOS])
df_extraccion

In [ ]:
# Verdad de referencia: los estados financieros que diligenciaste en la sección 1.4
ultimo = FINANCIEROS.loc[ANIO_ACTUAL] if ANIO_ACTUAL in FINANCIEROS.index else FINANCIEROS.iloc[-1]
verdad = ultimo.to_dict()
if pd.notna(verdad.get("impuestos")) and pd.notna(verdad.get("utilidad_antes_impuestos")):
    verdad["tasa_impuestos"] = 100 * verdad["impuestos"] / verdad["utilidad_antes_impuestos"]


def estado(fila):
    real = verdad.get(fila["campo"])
    if pd.isna(fila["valor_llm"]):
        return "— no encontrado"
    if real is None or pd.isna(real):
        return "sin dato de referencia"
    return "✅ coincide" if abs(fila["valor_llm"] / real - 1) < 0.01 else "⚠️ difiere"


df_extraccion["valor_referencia"] = df_extraccion["campo"].map(verdad)
df_extraccion["error_pct"] = 100 * (df_extraccion["valor_llm"] / df_extraccion["valor_referencia"] - 1)
df_extraccion["estado"] = df_extraccion.apply(estado, axis=1)
TABLA_VERIFICACION = df_extraccion[["campo", "valor_texto", "unidad_llm", "valor_llm", "valor_referencia",
                                    "error_pct", "cita_verificada", "estado"]].round(2)
TABLA_VERIFICACION

**Errores típicos en informes colombianos** (además de los de la sesión 8: segmento en vez de total, año
anterior, capex proyectado):

- **Unidad equivocada**: leer "5,61 billones" como millones, o confundir el *billón* español ($10^{12}$)
  con el *billion* inglés ($10^9$). El error es de tres o seis órdenes de magnitud.
- **Separador decimal**: devolver `5.610` como 5,61.
- **EBITDA por utilidad operacional**: los informes de gestión colombianos destacan el EBITDA, y el modelo
  lo toma como si fuera la utilidad operacional.
- **Cifra separada vs. consolidada**: muchos emisores presentan los dos juegos de estados financieros.
- **Cifras "ajustadas" o "proforma"** que no coinciden con los estados financieros auditados.
- La cifra **no está** en la sección de gestión (está solo en los estados financieros o en una gráfica).

### ✍️ Entregable 2 — Tabla de verificación y errores del LLM

*La tabla es `TABLA_VERIFICACION`. Reemplaza este texto con un párrafo (8 a 12 líneas) que responda:*
¿cuántos campos coincidieron?, ¿qué tipo de error hubo en cada campo que difiere (revisa `valor_texto`,
`unidad_llm` y `cita`)?, ¿alguna cita no se pudo verificar?, ¿el error fue de **recuperación** (el chunk
correcto no llegó al LLM) o de **lectura** (llegó, pero el modelo leyó mal)?, ¿qué cambiarías (consulta,
prompt, modelo, páginas) y qué pasó cuando lo probaste?

<a id="sec6"></a>
## 6. Aplicación: un DCF por escenarios informado por el texto

La división de responsabilidades es la de la sesión 8, con la fuente de cifras adaptada:

| Elemento | Fuente | Quién decide |
|---|---|---|
| Cifras históricas | Estados financieros consolidados (sección 1.4) | Dato, verificado por el analista |
| Guía de crecimiento, presión sobre márgenes, riesgos clave | Texto, leído por el LLM **con citas verificadas** | LLM (verificable) |
| Cómo se traduce esa lectura en números del DCF | Tabla `REGLAS` | **Analista** (explícito y discutible) |
| WACC base (en COP) y crecimiento terminal | Curso de valoración | **Analista** |

### 6.1 Históricos

In [ ]:
requeridas = ["ingresos", "utilidad_operativa", "utilidad_antes_impuestos", "impuestos", "dep_amort",
              "capex", "caja", "deuda", "acciones"]
assert not FINANCIEROS[requeridas].tail(2).isna().values.any(), (
    "Faltan cifras en FINANCIEROS: completa FINANCIEROS_MANUALES (sección 1.4) antes de construir el DCF.")

hist = FINANCIEROS.copy()
hist["crecimiento"] = hist["ingresos"].pct_change()
hist["margen_operativo"] = hist["utilidad_operativa"] / hist["ingresos"]
hist["tasa_impuestos"] = hist["impuestos"] / hist["utilidad_antes_impuestos"]
hist["da_pct_ventas"] = hist["dep_amort"] / hist["ingresos"]
hist["capex_pct_ventas"] = hist["capex"] / hist["ingresos"]
hist[["ingresos", "crecimiento", "margen_operativo", "tasa_impuestos", "da_pct_ventas", "capex_pct_ventas"]].round(4)

### 6.2 Lectura cualitativa del LLM

La misma lectura de analista de la sesión 8, con citas literales que se verifican contra el informe. Si la
administración no da guía numérica (frecuente en Colombia), `guia_crecimiento_cita` debe quedar vacía y
el DCF parte del crecimiento histórico.

In [ ]:
PROMPT_LECTURA = (
    "Eres un analista de valoración de empresas. Lee los fragmentos del informe anual y responde SOLO con "
    "un objeto JSON válido con estas claves:\n"
    '  "guia_crecimiento_cita": oración literal donde la administración da su expectativa de crecimiento de '
    'ingresos para el próximo año, o "" si no existe\n'
    '  "crecimiento_min_pct" y "crecimiento_max_pct": rango numérico en % que implica esa guía '
    '(por ejemplo "entre 3% y 5%" equivale a 3 y 5; "un dígito bajo" equivale aproximadamente a 1 y 4), '
    "o null si no hay guía\n"
    '  "presion_margen": "alta", "media" o "baja", según lo que digan los fragmentos sobre márgenes\n'
    '  "presion_margen_cita": oración literal que justifica presion_margen\n'
    '  "riesgos_clave": lista de hasta 3 objetos {"riesgo": resumen breve en español, "cita": oración literal}\n'
    "Usa solo información de los fragmentos."
)

ctx_lectura = pd.concat([
    recuperar("perspectivas: crecimiento esperado de ingresos y margen para el próximo año", k=3, seccion="gestion"),
    recuperar("el margen disminuyó por mayores costos, presión sobre la rentabilidad", k=2, seccion="gestion"),
    recuperar("podría afectar de manera material y adversa nuestra rentabilidad", k=3, seccion="riesgos"),
]).drop_duplicates("id")

mensajes = [{"role": "system", "content": PROMPT_LECTURA},
            {"role": "user", "content": f"Fragmentos:\n{formatear_contexto(ctx_lectura)}"}]
lectura = extraer_json(llamar_llm(mensajes, max_new_tokens=600)) or {}

texto_reporte = " ".join(TEXTOS[ANIO_ACTUAL].values())
guia_ok = cita_existe(lectura.get("guia_crecimiento_cita", ""), texto_reporte)
margen_ok = cita_existe(lectura.get("presion_margen_cita", ""), texto_reporte)
riesgos_ok = [r for r in lectura.get("riesgos_clave", []) if cita_existe(r.get("cita", ""), texto_reporte)]

print(json.dumps(lectura, indent=2, ensure_ascii=False))
print(f"\nCita de guía verificada: {guia_ok} | cita de margen verificada: {margen_ok} | "
      f"riesgos con cita verificada: {len(riesgos_ok)} de {len(lectura.get('riesgos_clave', []))}")

### 6.3 De la lectura a los supuestos: **tus** reglas

La mecánica es la de la sesión 8 (crecimiento desde la guía, margen ajustado por presión, prima de WACC
proporcional a `PROP_NUEVAS_RIESGOS`). Los valores de abajo son **un punto de partida ilustrativo para una
empresa colombiana valorada en pesos**; el taller pide que los reemplaces por los tuyos y los defiendas:

- `wacc_base`: en COP nominales. Debe ser coherente con tu curso de valoración (tasa libre de riesgo,
  prima de riesgo país, beta del sector, costo de la deuda y estructura de capital de **esta** empresa).
- `g_terminal`: en COP nominales no debería superar el crecimiento nominal de largo plazo de la economía
  (como referencia, la meta de inflación del Banco de la República es 3 %).
- `nwc_pct_delta_ventas`: mídelo con el capital de trabajo histórico de la empresa.
- Ajustes de margen, prima por riesgos nuevos y escenarios: ¿por qué esos tamaños y no otros?

In [ ]:
REGLAS = {   # ✍️ reemplaza por tu versión y justifícala en el Entregable 3
    "wacc_base": 0.120,                    # WACC en COP nominales
    "g_terminal": 0.035,                   # crecimiento terminal en COP nominales
    "nwc_pct_delta_ventas": 0.10,          # inversión en capital de trabajo = 10% del aumento en ventas
    "ajuste_margen": {"alta": -0.005, "media": -0.0025, "baja": 0.0},
    "prima_max_riesgos_nuevos": 0.005,     # hasta +50 pb de WACC…
    "prop_nuevas_para_prima_max": 0.20,    # …si ≥ 20% de las oraciones de riesgos son nuevas
    "escenarios": {   # (ajuste crecimiento, ajuste margen extra, ajuste WACC)
        "pesimista": (-0.01, -0.010, +0.010),
        "base":      (0.00,  0.000,  0.000),
        "optimista": (0.00,  None,  -0.005),   # None = sin presión de margen (vuelve al último margen)
    },
}

u = hist.iloc[-1]
g_hist = float(u["crecimiento"])
if guia_ok and lectura.get("crecimiento_min_pct") is not None and lectura.get("crecimiento_max_pct") is not None:
    g_min, g_max = float(lectura["crecimiento_min_pct"]) / 100, float(lectura["crecimiento_max_pct"]) / 100
    origen_g = "guía de la administración (verificada)"
else:
    g_min, g_max = g_hist - 0.01, g_hist + 0.01
    origen_g = "crecimiento histórico ±1 pp (no hubo guía verificada)"

presion = lectura.get("presion_margen") if margen_ok else "media"
prima_texto = REGLAS["prima_max_riesgos_nuevos"] * min(1, PROP_NUEVAS_RIESGOS / REGLAS["prop_nuevas_para_prima_max"])

supuestos = {}
for nombre, (aj_g, aj_m, aj_w) in REGLAS["escenarios"].items():
    g1 = {"pesimista": g_min + aj_g, "base": (g_min + g_max) / 2, "optimista": g_max}[nombre]
    margen = u["margen_operativo"] + (0 if aj_m is None else REGLAS["ajuste_margen"].get(presion, 0) + aj_m)
    supuestos[nombre] = {
        "crecimiento_anio1": g1,
        "margen_operativo": margen,
        "wacc": REGLAS["wacc_base"] + prima_texto + aj_w,
    }

print(f"Origen del crecimiento: {origen_g} | presión de margen: {presion} | "
      f"prima por riesgos nuevos: {prima_texto * 1e4:.0f} pb")
pd.DataFrame(supuestos).round(4)

### ✍️ Entregable 3 — Justificación de `REGLAS`

*Reemplaza este texto. Para cada parámetro que uses, escribe una línea con el valor, la fuente o el cálculo
y por qué ese valor y no otro:*

| Parámetro | Tu valor | Justificación (fuente, cálculo, criterio) |
|---|---|---|
| `wacc_base` | | |
| `g_terminal` | | |
| `nwc_pct_delta_ventas` | | |
| `ajuste_margen` | | |
| `prima_max_riesgos_nuevos` / `prop_nuevas_para_prima_max` | | |
| `escenarios` | | |

*Cierra con dos o tres líneas:* ¿la prima por riesgos nuevos es razonable dado lo que encontraste en el
Entregable 1 (riesgos genuinos vs. cambios de formato)? Si `PROP_NUEVAS_RIESGOS` está inflada por un cambio
de diagramación, ¿cómo lo corregiste?

### 6.4 El DCF

$FCFF_t = EBIT_t(1-t) + D\&A_t - Capex_t - \Delta NWC_t$, cinco años explícitos y valor terminal de
Gordon. Frente a la sesión 8 hay un ajuste: del valor de la firma se restan la deuda **y las
participaciones no controladoras** (`minoritarios`), importantes en los grupos empresariales colombianos
que consolidan filiales que no poseen al 100 %. Como todo está en millones de COP y las acciones en
millones, el valor por acción queda en **COP**.

In [ ]:
def dcf(ingresos_0, crecimiento_anio1, margen_operativo, wacc, g_terminal, tasa_impuestos,
        da_pct, capex_pct, nwc_pct, caja, deuda, minoritarios, acciones, anios=5):
    crecimientos = np.linspace(crecimiento_anio1, g_terminal, anios)
    filas, ingresos_prev = [], ingresos_0
    for t, g in enumerate(crecimientos, start=1):
        ingresos = ingresos_prev * (1 + g)
        ebit = ingresos * margen_operativo
        fcff = ebit * (1 - tasa_impuestos) + ingresos * da_pct - ingresos * capex_pct - nwc_pct * (ingresos - ingresos_prev)
        filas.append({"anio": t, "crecimiento": g, "ingresos": ingresos, "ebit": ebit, "fcff": fcff,
                      "factor_desc": 1 / (1 + wacc) ** t})
        ingresos_prev = ingresos
    proy = pd.DataFrame(filas).set_index("anio")
    vp_flujos = (proy["fcff"] * proy["factor_desc"]).sum()
    valor_terminal = proy["fcff"].iloc[-1] * (1 + g_terminal) / (wacc - g_terminal)
    vp_terminal = valor_terminal * proy["factor_desc"].iloc[-1]
    valor_firma = vp_flujos + vp_terminal
    patrimonio = valor_firma + caja - deuda - minoritarios
    return {"valor_firma": valor_firma, "vp_terminal_pct": vp_terminal / valor_firma,
            "patrimonio": patrimonio, "valor_por_accion": patrimonio / acciones}, proy


PARAMS_COMUNES = dict(
    ingresos_0=u["ingresos"], g_terminal=REGLAS["g_terminal"],
    tasa_impuestos=hist["tasa_impuestos"].mean(), da_pct=hist["da_pct_ventas"].mean(),
    capex_pct=hist["capex_pct_ventas"].mean(), nwc_pct=REGLAS["nwc_pct_delta_ventas"],
    caja=u["caja"], deuda=u["deuda"], minoritarios=u["minoritarios"], acciones=u["acciones"],
)

resultados, proyecciones = {}, {}
for nombre, s in supuestos.items():
    resultados[nombre], proyecciones[nombre] = dcf(**PARAMS_COMUNES, **s)

df_valor = pd.DataFrame(resultados).T
df_valor["vs_precio_mercado"] = df_valor["valor_por_accion"] / PRECIO_MERCADO - 1 if PRECIO_MERCADO else np.nan
display(df_valor.round(3))
display(Markdown("**Proyección del escenario base (millones de COP):**"))
proyecciones["base"].round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (a) Rango de valor por escenario vs precio de mercado
orden = ["pesimista", "base", "optimista"]
axes[0].barh(orden, df_valor.loc[orden, "valor_por_accion"], color=["#c0504d", "#7f7f7f", "#4f81bd"])
if PRECIO_MERCADO:
    axes[0].axvline(PRECIO_MERCADO, color="black", ls="--", label=f"Precio de mercado: {PRECIO_MERCADO:,.0f}")
    axes[0].legend(loc="lower right")
for i, v in enumerate(df_valor.loc[orden, "valor_por_accion"]):
    axes[0].text(v, i, f" {v:,.0f}", va="center")
axes[0].set_title("Valor por acción por escenario")
axes[0].set_xlabel("COP por acción")

# (b) Sensibilidad del escenario base a WACC y g
waccs = np.round(supuestos["base"]["wacc"] + np.arange(-0.02, 0.0201, 0.01), 4)
gs = np.round(REGLAS["g_terminal"] + np.arange(-0.01, 0.0101, 0.005), 4)
tabla = pd.DataFrame(
    [[dcf(**{**PARAMS_COMUNES, "g_terminal": g},
          crecimiento_anio1=supuestos["base"]["crecimiento_anio1"],
          margen_operativo=supuestos["base"]["margen_operativo"], wacc=w)[0]["valor_por_accion"]
      for g in gs] for w in waccs],
    index=[f"{w:.1%}" for w in waccs], columns=[f"{g:.1%}" for g in gs],
)
sns.heatmap(tabla, annot=True, fmt=",.0f", cmap="RdBu", center=PRECIO_MERCADO, ax=axes[1], cbar=False)
axes[1].set_title("Sensibilidad (base): valor por acción en COP")
axes[1].set_xlabel("crecimiento terminal")
axes[1].set_ylabel("WACC")
plt.tight_layout()
plt.show()

### 6.5 Borrador de la nota de valoración (lo redacta la IA)

El LLM redacta a partir de un resumen estructurado y comprobamos que no agregue cifras. Guardamos el
borrador en `NOTA_BORRADOR` porque el taller pide que muestres **qué cambiaste**.

In [ ]:
resumen_valoracion = {
    "empresa": EMPRESA,
    "anio": int(ANIO_ACTUAL),
    "moneda": "COP",
    "precio_mercado": None if PRECIO_MERCADO is None else round(PRECIO_MERCADO),
    "valor_por_accion": {k: round(float(v)) for k, v in df_valor["valor_por_accion"].items()},
    "supuestos": {k: {kk: f"{vv:.1%}" for kk, vv in v.items()} for k, v in supuestos.items()},
    "crecimiento_ultimo_anio": f"{g_hist:.1%}",
    "margen_operativo_ultimo_anio": f"{u['margen_operativo']:.1%}",
    "origen_crecimiento": origen_g,
    "guia_administracion": lectura.get("guia_crecimiento_cita", "") if guia_ok else "",
    "presion_margen": presion,
    "riesgos_clave": [r["riesgo"] for r in riesgos_ok],
    "proporcion_oraciones_nuevas_en_riesgos": f"{PROP_NUEVAS_RIESGOS:.0%}",
}

PROMPT_NOTA = (
    "Eres un analista de valoración. Redacta en español una nota de valoración de máximo 200 palabras con "
    "tres párrafos: (1) desempeño reciente y guía de la administración, (2) riesgos y cómo se reflejan en los "
    "escenarios, (3) comparación del rango de valor por acción, en pesos colombianos, con el precio de mercado. "
    "Usa EXCLUSIVAMENTE las cifras del JSON que se te entrega; no agregues ninguna otra cifra. "
    "No hagas recomendaciones de compra o venta: presenta la conclusión como análisis."
)

NOTA_BORRADOR = llamar_llm([{"role": "system", "content": PROMPT_NOTA},
                            {"role": "user", "content": json.dumps(resumen_valoracion, ensure_ascii=False)}],
                           max_new_tokens=500)
display(Markdown(NOTA_BORRADOR))

sin_soporte = numeros_no_soportados(NOTA_BORRADOR, json.dumps(resumen_valoracion, ensure_ascii=False))
print("\nCifras del borrador que NO están en los datos de entrada:", sin_soporte or "ninguna ✅")

<a id="sec7"></a>
## 7. ✍️ Entregable 4 — Nota de valoración final, editada y firmada

Copia el borrador en `NOTA_FINAL` y **edítalo**: corrige cifras sin soporte, matiza lo que el modelo
exageró, agrega el contexto que solo tú conoces (sector, coyuntura colombiana, lo que viste al leer los
riesgos nuevos) y firma. La celda siguiente muestra las diferencias entre el borrador y tu versión y
vuelve a comprobar las cifras.

In [ ]:
ANALISTA = "Nombre Apellido"        # ✍️ tu nombre
FECHA_ANALISIS = "AAAA-MM-DD"       # ✍️ fecha del análisis (y del precio de mercado)

NOTA_FINAL = """
Pega aquí el borrador y edítalo.
"""

In [ ]:
diferencias = [l for l in difflib.unified_diff(
    [o.strip() for o in re.split(r"(?<=[.!?])\s+", normalizar_espacios(NOTA_BORRADOR))],
    [o.strip() for o in re.split(r"(?<=[.!?])\s+", normalizar_espacios(NOTA_FINAL))],
    fromfile="borrador IA", tofile="nota final", lineterm="", n=0) if not l.startswith("@@")]

display(Markdown(f"### Nota de valoración — {EMPRESA} ({ANIO_ACTUAL})\n\n{NOTA_FINAL.strip()}\n\n"
                 f"*Elaboró: {ANALISTA} — {FECHA_ANALISIS}. Borrador redactado con asistencia de IA "
                 f"({LLM_MODEL_NAME if BACKEND_LLM == 'local' else API_MODEL}); cifras y supuestos verificados por el analista.*"))

sin_soporte_final = numeros_no_soportados(NOTA_FINAL, json.dumps(resumen_valoracion, ensure_ascii=False))
print("Cifras de la nota final que no están en resumen_valoracion (debes poder justificar cada una):",
      sin_soporte_final or "ninguna ✅")
print("\nCambios frente al borrador de la IA (− eliminado, + agregado):")
if "Pega aquí el borrador" in NOTA_FINAL or not diferencias:
    print("⚠️  Todavía no has editado la nota: el taller pide tu versión, no el borrador.")
else:
    print("\n".join(diferencias))

*Reemplaza este texto con 5 a 8 líneas:* **¿qué cambiaste del borrador de la IA y por qué?** (cifras
corregidas, afirmaciones que suavizaste o eliminaste, contexto que agregaste). ¿Qué parte del análisis
**no** habrías podido delegar en el modelo?

### Lista de chequeo antes de entregar

- [ ] El cuaderno corre de principio a fin con `FUENTE = "pdf"` y **no** cayó al plan B.
- [ ] `PAGINAS` tiene los rangos que tú verificaste para los dos años (no la selección automática).
- [ ] `FINANCIEROS` no tiene vacíos y cada cifra viene de los estados financieros **consolidados**.
- [ ] Entregable 1: tono y oraciones nuevas comentados (sección 3).
- [ ] Entregable 2: `TABLA_VERIFICACION` y párrafo de errores (sección 5).
- [ ] Entregable 3: tu `REGLAS` y su justificación (sección 6.3).
- [ ] Entregable 4: nota final editada y firmada, con los cambios explicados (sección 7).

### Limitaciones que debes tener presentes

- Con solo dos informes y una empresa, el tono y las oraciones nuevas son **indicios para leer mejor**, no
  evidencia estadística.
- La calidad de todo el flujo depende de la extracción del PDF: una sección mal delimitada contamina el
  tono, la comparación entre años y el RAG.
- El DCF por FCFF de este cuaderno no aplica a entidades financieras, y simplifica aspectos que pueden ser
  materiales (arrendamientos NIIF 16, inversiones en asociadas, pasivos pensionales, descuento de *holding*).
- Esto es un ejercicio académico: no constituye una recomendación de inversión.